# GamePulse — 03 · Gold Layer
## Joins, Aggregations & Analysis-Ready Tables

**SYST52461 – Big Data Storage and Analysis**  
**Term Project · Winter 2026**

---

### Purpose
This notebook reads the four Silver tables, joins and aggregates them into three **Gold** Delta tables  
that are optimized for dashboard visualizations and EDA.

### Gold Tables Produced
| Table | Description |
|---|---|
| `player_summary_gold` | One row per player — lifetime sessions, total playtime, total spend, avg score |
| `game_engagement_gold` | One row per game — total sessions, avg session duration, avg score, win rate |
| `revenue_trend_gold` | Monthly aggregation — total revenue, total sessions, and unique active players |

## 1. Environment Setup

In [ ]:
import logging
from pyspark.sql import functions as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("gamepulse.gold")

CATALOG = "sandbox_catalog"
SCHEMA  = "gamepulse"

log.info("Gold notebook ready.")

## 2. Helper Utilities

In [ ]:
def read_table(table_name: str):
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df = spark.read.table(full_name)
    log.info("Loaded  <- %s  (%d rows)", full_name, df.count())
    return df


def write_table(df, table_name: str) -> None:
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    log.info("Written -> %s  (%d rows)", full_name, df.count())


log.info("Helpers defined.")

## 3. Load Silver Tables

In [ ]:
players_s   = read_table("players_silver")
games_s     = read_table("games_silver")
sessions_s  = read_table("game_sessions_silver")
purchases_s = read_table("purchases_silver")

---
## 4. Build `player_summary_gold`

**Logic:**
1. Aggregate `game_sessions_silver` by `player_id` → lifetime session stats
2. Aggregate `purchases_silver` by `player_id` → lifetime spend stats
3. Join both aggregations to `players_silver` demographics

**Key columns produced:**
- `total_sessions`, `total_playtime_min`, `avg_score`, `win_rate`
- `total_purchases`, `total_spent`, `avg_purchase_value`
- `region`, `account_type`, `age`

In [ ]:
# ── Step 1: Session aggregation per player ─────────────────────────────────────
session_agg = (
    sessions_s
    .groupBy("player_id")
    .agg(
        F.count("session_id").alias("total_sessions"),
        F.round(F.sum("duration_minutes"), 1).alias("total_playtime_min"),
        F.round(F.avg("score"), 1).alias("avg_score"),
        F.round(
            F.count(F.when(F.col("outcome") == "Win", True)) /
            F.count("session_id") * 100,
            1
        ).alias("win_rate_pct"),
        F.max("session_date").alias("last_session_date"),
    )
)

log.info("session_agg: %d player rows", session_agg.count())

In [ ]:
# ── Step 2: Purchase aggregation per player ────────────────────────────────────
purchase_agg = (
    purchases_s
    .groupBy("player_id")
    .agg(
        F.count("purchase_id").alias("total_purchases"),
        F.round(F.sum("amount_spent"), 2).alias("total_spent"),
        F.round(F.avg("amount_spent"), 2).alias("avg_purchase_value"),
        F.max("purchase_date").alias("last_purchase_date"),
    )
)

log.info("purchase_agg: %d player rows", purchase_agg.count())

In [ ]:
# ── Step 3: Join everything to player demographics ─────────────────────────────
player_summary_gold = (
    players_s
    .select("player_id", "username", "age", "region", "account_type", "join_date")
    .join(session_agg,  on="player_id", how="left")
    .join(purchase_agg, on="player_id", how="left")
    # Players with no sessions or purchases get 0 instead of null
    .fillna({
        "total_sessions":    0,
        "total_playtime_min": 0.0,
        "avg_score":          0.0,
        "win_rate_pct":       0.0,
        "total_purchases":   0,
        "total_spent":        0.0,
        "avg_purchase_value": 0.0,
    })
)

print(f"player_summary_gold: {player_summary_gold.count()} rows")
display(player_summary_gold.orderBy("total_spent", ascending=False).limit(15))

---
## 5. Build `game_engagement_gold`

**Logic:**
1. Aggregate `game_sessions_silver` by `game_id` → engagement metrics
2. Join to `games_silver` for game title, genre, and mode

In [ ]:
# Aggregate session metrics by game
game_agg = (
    sessions_s
    .groupBy("game_id")
    .agg(
        F.count("session_id").alias("total_sessions"),
        F.countDistinct("player_id").alias("unique_players"),
        F.round(F.avg("duration_minutes"), 1).alias("avg_duration_min"),
        F.round(F.avg("score"), 1).alias("avg_score"),
        F.round(F.sum("duration_minutes"), 1).alias("total_playtime_min"),
        F.round(
            F.count(F.when(F.col("outcome") == "Win", True)) /
            F.count("session_id") * 100,
            1
        ).alias("win_rate_pct"),
    )
)

# Join to game metadata
game_engagement_gold = (
    games_s
    .select("game_id", "title", "genre", "game_mode", "release_year")
    .join(game_agg, on="game_id", how="left")
    .fillna({"total_sessions": 0, "unique_players": 0})
    .orderBy("total_sessions", ascending=False)
)

print(f"game_engagement_gold: {game_engagement_gold.count()} rows")
display(game_engagement_gold)

---
## 6. Build `revenue_trend_gold`

**Logic:**  
Aggregate `purchases_silver` and `game_sessions_silver` by calendar month  
to produce a monthly time series of revenue and engagement activity.

In [ ]:
# Monthly revenue from purchases
monthly_revenue = (
    purchases_s
    .withColumn("month",  F.date_format("purchase_date", "yyyy-MM"))
    .groupBy("month")
    .agg(
        F.round(F.sum("amount_spent"), 2).alias("total_revenue"),
        F.count("purchase_id").alias("total_purchases"),
        F.countDistinct("player_id").alias("paying_players"),
    )
)

# Monthly session activity
monthly_sessions = (
    sessions_s
    .withColumn("month", F.date_format("session_date", "yyyy-MM"))
    .groupBy("month")
    .agg(
        F.count("session_id").alias("total_sessions"),
        F.countDistinct("player_id").alias("active_players"),
        F.round(F.avg("duration_minutes"), 1).alias("avg_session_duration_min"),
    )
)

# Join on month
revenue_trend_gold = (
    monthly_revenue
    .join(monthly_sessions, on="month", how="outer")
    .orderBy("month")
)

print(f"revenue_trend_gold: {revenue_trend_gold.count()} rows (months)")
display(revenue_trend_gold)

---
## 7. Write Gold Tables to Delta

In [ ]:
write_table(player_summary_gold,  "player_summary_gold")
write_table(game_engagement_gold, "game_engagement_gold")
write_table(revenue_trend_gold,   "revenue_trend_gold")

log.info("All Gold tables written successfully.")

---
## 8. Gold Layer Summary

In [ ]:
print("=" * 50)
print("Gold Layer Summary")
print("=" * 50)
for tbl in ["player_summary_gold", "game_engagement_gold", "revenue_trend_gold"]:
    df = spark.read.table(f"{CATALOG}.{SCHEMA}.{tbl}")
    print(f"  {tbl:<30} {df.count():>5} rows")
print("=" * 50)